# 構造化出力とエージェンティックワークフロー


In [ ]:
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

## 構造化出力（Structured Outputs）


### Chat Completions APIの構造化出力


In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field


class SpamJudgeResult(BaseModel):
    is_spam: bool = Field(description="スパムかどうか")
    reason_in_japanese: str = Field(description="判定理由")


def spam_judge(inquiry: str) -> SpamJudgeResult:
    client = OpenAI()
    response = client.chat.completions.parse(
        model="gpt-5.6-luna",
        messages=[
            {
                "role": "developer",
                "content": "以下のお問い合わせがスパムか判定してください。",
            },
            {
                "role": "user",
                "content": f"お問い合わせ: {inquiry}",
            },
        ],
        reasoning_effort="none",
        response_format=SpamJudgeResult,
    )

    return response.choices[0].message.parsed


result = spam_judge("AIエージェントの開発を依頼したいです。")
print(type(result))
print(result.model_dump_json(indent=2))


### LangChainのwith_structured_outputを使った構造化出力


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.chat_models import init_chat_model


class SpamJudgeResult(BaseModel):
    is_spam: bool = Field(description="スパムかどうか")
    reason_in_japanese: str = Field(description="判定理由")


def spam_judge(inquiry: str) -> SpamJudgeResult:
    model = init_chat_model(
        model="gpt-5.6-luna",
        model_provider="openai",
        reasoning_effort="none",
    )
    model_with_structure = model.with_structured_output(SpamJudgeResult)

    prompt = [
        SystemMessage(content="以下のお問い合わせがスパムか判定してください。"),
        HumanMessage(content=f"お問い合わせ: {inquiry}"),
    ]
    return model_with_structure.invoke(prompt)


result = spam_judge("AIエージェントの開発を依頼したいです。")
print(type(result))
print(result.model_dump_json(indent=2))

## 返信文生成ワークフロー


In [ ]:
from typing_extensions import TypedDict


class State(TypedDict):
    # 入力
    inquiry: str
    # スパム判定の出力
    is_spam: bool
    reason_in_japanese: str
    # 返信文生成の出力
    response: str | None

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gpt-5.6-luna",
    model_provider="openai",
    reasoning_effort="none",
)

In [ ]:
from typing import Any
from pydantic import BaseModel, Field


class SpamJudgeResult(BaseModel):
    is_spam: bool = Field(description="スパムかどうか")
    reason_in_japanese: str = Field(description="判定理由")


def spam_judge_node(state: State) -> dict[str, Any]:
    model_with_structure = model.with_structured_output(SpamJudgeResult)
    prompt = [
        SystemMessage(content="以下のお問い合わせがスパムか判定してください。"),
        HumanMessage(content=f"お問い合わせ: {state['inquiry']}"),
    ]
    return model_with_structure.invoke(prompt)

In [ ]:
def generate_answer_node(state: State) -> dict[str, Any]:
    prompt = [
        SystemMessage(content="以下のお問い合わせに対する、返信文を出力してください。"),
        HumanMessage(content=f"お問い合わせ: {state['inquiry']}"),
    ]
    ai_message = model.invoke(prompt)
    return {"response": ai_message.content}


In [ ]:
from langgraph.graph import StateGraph, START, END

graph_builder = StateGraph(State)

graph_builder.add_node("spam_judge_node", spam_judge_node)
graph_builder.add_node("generate_answer_node", generate_answer_node)

graph_builder.add_edge(START, "spam_judge_node")

# checkノードから次のノードへの遷移に条件付きエッジを定義
# state.is_spamの値がTrueならENDノードへ、Falseならgenerate_answer_nodeへ
graph_builder.add_conditional_edges(
    "spam_judge_node",
    lambda state: state["is_spam"],
    {True: END, False: "generate_answer_node"},
)

graph_builder.add_edge("generate_answer_node", END)

graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
import json

initial_state = {"inquiry": "AIエージェントの開発を依頼したいです。"}
final_state = graph.invoke(initial_state)
print(json.dumps(final_state, ensure_ascii=False, indent=2))